In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, List, Optional, Sequence
from itertools import combinations

import pandas as pd
import numpy as np
from sqlalchemy.engine import Engine

from datetime import datetime, timedelta

from utils.postgres_util import (
    sanitize_sql_identifier, 
    read_sql_dataframe,
    get_engine_from_env,
)

from utils.file_io import (
    save_data,
    load_data,
)

from utils.paths import get_paths

from utils.pipeline_config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

---

In [2]:
# Get Path's Object
paths = get_paths()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CONFIG_ROOT = paths.configs

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CONFIG_STAGE = "synthetic"
CONFIG_DATASET = "pump"
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

CONFIG = load_pipeline_config(
    config_root=CONFIG_ROOT,
    stage=CONFIG_STAGE,
    dataset=CONFIG_DATASET,
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    project_root=paths.root,
).data

SYN_CFG = CONFIG["synthetic"]
PATHS = CONFIG["resolved_paths"]
FILENAMES = CONFIG["filenames"]

SYNTHETIC_DATA_PATH = Path(PATHS["data_synthetic_dir"])

SCHEMA = str("capstone")
DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

CHUNK_SIZE = 100_000
ORDER_BY = "observation_timestamp"


SOURCE_TABLE_NAME = "synthetic_observations_timestamped_stage"

OUTPUT_DIRECTORY = SYNTHETIC_DATA_PATH
OUTPUT_BASE_FILE_NAME = "synthetic_timestamped_export"

In [ ]:
current_datetime = datetime.now()
adjusted_time = current_datetime - timedelta(hours=4)

formatted_datetime = adjusted_time.strftime("%m%d%Y_%H%M")

---

In [4]:
def export_synthetic_table_to_csv_parts(
    engine,
    *,
    schema: str = "capstone",
    table_name: str = "synthetic_observations_timestamped_stage",
    output_dir: str | Path = "/workspace/artifacts/exports",
    base_file_name: str = "synthetic_timestamped_export",
    chunk_size: int = 100_000,
    order_by: Optional[str] = "observation_timestamp",
    timestamp,
) -> List[Path]:
    """
    Read a synthetic Postgres table using the requested projection/machine_status mapping
    and export the result into multiple CSV parts.

    Returns
    -------
    List[Path]
        Paths to the CSV part files that were written.
    """
    safe_schema = sanitize_sql_identifier(schema)
    safe_table = sanitize_sql_identifier(table_name)
    safe_order_by = sanitize_sql_identifier(order_by) if order_by else None

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    sensor_columns = [f"sensor_{i:02d}" for i in range(52)]
    sensor_select_sql = ",\n        ".join(sensor_columns)

    order_clause = f'\n    ORDER BY "{safe_order_by}"' if safe_order_by else ""

    count_sql = f"""
    SELECT COUNT(*) AS row_count
    FROM "{safe_schema}"."{safe_table}"
    """

    total_rows_df = read_sql_dataframe(engine, count_sql)
    total_rows = int(total_rows_df.loc[0, "row_count"])

    if total_rows == 0:
        print(f"No rows found in {safe_schema}.{safe_table}")
        return []

    created_files: List[Path] = []

    for offset in range(0, total_rows, chunk_size):
        sql = f"""
        SELECT
            dataset_id,
            run_id,
            asset_id,
            observation_timestamp AS timestamp,
            {sensor_select_sql},
            CASE LOWER(COALESCE(phase, 'normal'))
                WHEN 'normal' THEN 'NORMAL'
                WHEN 'broken' THEN 'BROKEN'
                WHEN 'abnormal' THEN 'BROKEN'
                WHEN 'failure' THEN 'BROKEN'
                WHEN 'failed' THEN 'BROKEN'
                WHEN 'fault' THEN 'BROKEN'
                WHEN 'recovering' THEN 'RECOVERING'
                WHEN 'recovery' THEN 'RECOVERING'
                ELSE 'NORMAL'
            END AS machine_status
        FROM "{safe_schema}"."{safe_table}"
        {order_clause}
        LIMIT {int(chunk_size)} OFFSET {int(offset)}
        """

        chunk_df = read_sql_dataframe(engine, sql)

        if chunk_df.empty:
            continue

        part_number = (offset // chunk_size) + 1
        part_file_name = f"{base_file_name}_{timestamp}_part_{part_number:03d}.csv"

        output_path = save_data(
            chunk_df,
            output_dir,
            part_file_name,
            index=False,
        )

        created_files.append(output_path)
        print(
            f"[export] wrote part {part_number:03d} | "
            f"rows={len(chunk_df):,} | path={output_path}"
        )

    return created_files

---

In [5]:
engine = get_engine_from_env()

---

In [ ]:
print(f"Starting Export at {formatted_datetime}")

Starting Export


In [7]:
exported_files = export_synthetic_table_to_csv_parts(
    engine,
    schema=SCHEMA,
    table_name=SOURCE_TABLE_NAME,
    output_dir=OUTPUT_DIRECTORY,
    base_file_name=OUTPUT_BASE_FILE_NAME,
    chunk_size=CHUNK_SIZE,
    order_by=ORDER_BY,
    timestamp=formatted_datetime,
)

exported_files

[export] wrote part 001 | rows=100,000 | path=/workspace/data/synthetic/synthetic_timestamped_export_04112026_2333_part_001.csv
[export] wrote part 002 | rows=100,000 | path=/workspace/data/synthetic/synthetic_timestamped_export_04112026_2333_part_002.csv
[export] wrote part 003 | rows=25,000 | path=/workspace/data/synthetic/synthetic_timestamped_export_04112026_2333_part_003.csv


[PosixPath('/workspace/data/synthetic/synthetic_timestamped_export_04112026_2333_part_001.csv'),
 PosixPath('/workspace/data/synthetic/synthetic_timestamped_export_04112026_2333_part_002.csv'),
 PosixPath('/workspace/data/synthetic/synthetic_timestamped_export_04112026_2333_part_003.csv')]

---

In [8]:
# Dispose Engine
engine.dispose()

In [ ]:
current_datetime = datetime.now()
adjusted_time = current_datetime - timedelta(hours=4)

formatted_datetime = adjusted_time.strftime("%m%d%Y_%H%M")

In [ ]:
print(f"Run export completed, files exported, at {formatted_datetime}")

Run export completed, file exported.


----

In [ ]:
def export_synthetic_table_to_csv(
    engine,
    *,
    schema: str = "capstone",
    table_name: str = "synthetic_observations_timestamped_stage",
    output_dir: str | Path = "/workspace/artifacts/exports",
    base_file_name: str = "synthetic_timestamped_export",
    order_by: Optional[str] = "observation_timestamp",
    timestamp: Optional[str] = None,
) -> Path | None:
    """
    Read a synthetic Postgres table using the requested projection/machine_status mapping
    and export the result into a single CSV file.

    Returns
    -------
    Path | None
        Path to the CSV file that was written, or None if no rows were found.
    """
    safe_schema = sanitize_sql_identifier(schema)
    safe_table = sanitize_sql_identifier(table_name)
    safe_order_by = sanitize_sql_identifier(order_by) if order_by else None

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if timestamp is None:
        timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    sensor_columns = [f"sensor_{i:02d}" for i in range(52)]
    sensor_select_sql = ",\n            ".join(sensor_columns)

    order_clause = f'\n        ORDER BY "{safe_order_by}"' if safe_order_by else ""

    count_sql = f"""
    SELECT COUNT(*) AS row_count
    FROM "{safe_schema}"."{safe_table}"
    """

    total_rows_df = read_sql_dataframe(engine, count_sql)
    total_rows = int(total_rows_df.loc[0, "row_count"])

    if total_rows == 0:
        print(f"No rows found in {safe_schema}.{safe_table}")
        return None

    '''
    sql = f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_timestamp AS timestamp,
        {sensor_select_sql},
        CASE LOWER(COALESCE(phase, 'normal'))
            WHEN 'normal' THEN 'NORMAL'
            WHEN 'broken' THEN 'BROKEN'
            WHEN 'abnormal' THEN 'BROKEN'
            WHEN 'failure' THEN 'BROKEN'
            WHEN 'failed' THEN 'BROKEN'
            WHEN 'fault' THEN 'BROKEN'
            WHEN 'recovering' THEN 'RECOVERING'
            WHEN 'recovery' THEN 'RECOVERING'
            ELSE 'NORMAL'
        END AS machine_status
    FROM "{safe_schema}"."{safe_table}"
    {order_clause}
    """
    '''
    
    sql = f"""
    SELECT
        observation_timestamp AS timestamp,
        {sensor_select_sql},
        CASE LOWER(COALESCE(phase, 'normal'))
            WHEN 'normal' THEN 'NORMAL'
            WHEN 'broken' THEN 'BROKEN'
            WHEN 'abnormal' THEN 'BROKEN'
            WHEN 'failure' THEN 'BROKEN'
            WHEN 'failed' THEN 'BROKEN'
            WHEN 'fault' THEN 'BROKEN'
            WHEN 'recovering' THEN 'RECOVERING'
            WHEN 'recovery' THEN 'RECOVERING'
            ELSE 'NORMAL'
        END AS machine_status
    FROM "{safe_schema}"."{safe_table}"
    {order_clause}
    """

    export_df = read_sql_dataframe(engine, sql)

    if export_df.empty:
        print(f"No rows returned from {safe_schema}.{safe_table}")
        return None

    file_name = f"{base_file_name}_{timestamp}.csv"

    output_path = save_data(
        export_df,
        output_dir,
        file_name,
        index=False,
    )

    print(
        f"[export] wrote single CSV | "
        f"rows={len(export_df):,} | path={output_path}"
    )

    return output_path

In [ ]:
export_path = export_synthetic_table_to_csv(
    engine,
    schema="capstone",
    table_name="synthetic_observations_timestamped_stage",
    output_dir="/workspace/artifacts/exports",
    base_file_name="synthetic_timestamped_export_full_",
    timestamp=formatted_datetime,
)

---

In [ ]:
current_datetime = datetime.now()
adjusted_time = current_datetime - timedelta(hours=4)

formatted_datetime = adjusted_time.strftime("%m%d%Y_%H%M")

In [ ]:
print(f"starting scorecard at {formatted_datetime}")

starting scorecard


In [11]:
engine = get_engine_from_env()

In [12]:
def load_source_dataframe_for_comparison(
    *,
    file_name: str,
    file_dir: Optional[str | Path] = None,
    use_data_raw_dir: bool = True,
    drop_unnamed_index_columns: bool = True,
    rename_columns: Optional[dict[str, str]] = None,
    sort_by: Optional[Sequence[str]] = None,
    reset_index_after_sort: bool = True,
    **read_kwargs: Any,
) -> pd.DataFrame:
    """
    Load a source CSV/Parquet for comparison using the project file_io pattern,
    but WITHOUT Bronze metadata stamping.
    """
    paths = get_paths()

    if file_dir is None:
        file_dir = paths.data_raw if use_data_raw_dir else paths.root

    dataframe = load_data(file_dir, file_name=file_name, **read_kwargs).copy()

    if drop_unnamed_index_columns:
        unnamed_columns = [
            column for column in dataframe.columns
            if str(column).strip().lower().startswith("unnamed:")
        ]
        if unnamed_columns:
            dataframe = dataframe.drop(columns=unnamed_columns)

    if rename_columns:
        dataframe = dataframe.rename(columns=rename_columns)

    if sort_by:
        valid_sort_columns = [column for column in sort_by if column in dataframe.columns]
        if valid_sort_columns:
            dataframe = dataframe.sort_values(valid_sort_columns)
            if reset_index_after_sort:
                dataframe = dataframe.reset_index(drop=True)

    return dataframe

In [13]:
def read_postgres_table_for_comparison(
    *,
    schema: str,
    table_name: str,
    selected_columns: Optional[Sequence[str]] = None,
    where_sql: Optional[str] = None,
    order_by: Optional[Sequence[str]] = None,
    limit: Optional[int] = None,
    engine: Optional[Engine] = None,
    params: Optional[dict[str, Any]] = None,
) -> pd.DataFrame:
    """
    Read a Postgres table into a dataframe for comparison work.
    """
    if engine is None:
        engine = get_engine_from_env()

    safe_schema = sanitize_sql_identifier(schema)
    safe_table = sanitize_sql_identifier(table_name)

    if selected_columns:
        cleaned_columns = [
            f'"{sanitize_sql_identifier(column)}"'
            for column in selected_columns
        ]
        select_sql = ", ".join(cleaned_columns)
    else:
        select_sql = "*"

    sql = f'SELECT {select_sql} FROM "{safe_schema}"."{safe_table}"'

    if where_sql is not None and str(where_sql).strip() != "":
        sql += f" WHERE {where_sql}"

    if order_by:
        cleaned_order = [
            f'"{sanitize_sql_identifier(column)}"'
            for column in order_by
        ]
        sql += f' ORDER BY {", ".join(cleaned_order)}'

    if limit is not None:
        sql += f" LIMIT {int(limit)}"

    dataframe = read_sql_dataframe(
        engine,
        sql,
        params=params or {},
    )

    return dataframe.copy()

In [ ]:


def read_synthetic_comparison_projection_dataframe(
    *,
    schema: str = "capstone",
    table_name: str = "synthetic_observations_timestamped_stage",
    sensor_columns: Optional[Sequence[str]] = None,
    where_sql: Optional[str] = None,
    order_by: Optional[Sequence[str]] = None,
    limit: Optional[int] = None,
    engine: Optional[Engine] = None,
    params: Optional[dict[str, Any]] = None,
) -> pd.DataFrame:
    """
    Read the synthetic Postgres table into the same comparison-ready shape used
    by the multipart export logic:
      - timestamp
      - sensor_* columns
      - machine_status (derived from phase)

    This should be the standard synthetic reader for scorecard/comparison work.
    """

    if engine is None:
        engine = get_engine_from_env()

    safe_schema = sanitize_sql_identifier(schema)
    safe_table = sanitize_sql_identifier(table_name)

    if sensor_columns is None:
        sensor_columns = [f"sensor_{i:02d}" for i in range(52)]

    sensor_select_sql = ",\n            ".join(
        f'"{sanitize_sql_identifier(column)}"'
        for column in sensor_columns
    )

    sql = f"""
        SELECT
            observation_timestamp AS timestamp,
            phase,
            {sensor_select_sql},
            CASE LOWER(COALESCE(phase, 'normal'))
                WHEN 'normal' THEN 'NORMAL'
                WHEN 'broken' THEN 'BROKEN'
                WHEN 'abnormal' THEN 'BROKEN'
                WHEN 'failure' THEN 'BROKEN'
                WHEN 'failed' THEN 'BROKEN'
                WHEN 'fault' THEN 'BROKEN'
                WHEN 'recovering' THEN 'RECOVERING'
                WHEN 'recovery' THEN 'RECOVERING'
                ELSE 'NORMAL'
            END AS machine_status
        FROM "{safe_schema}"."{safe_table}"
    """

    if where_sql is not None and str(where_sql).strip() != "":
        sql += f"\nWHERE {where_sql}"

    if order_by:
        cleaned_order = [
            f'"{sanitize_sql_identifier(column)}"'
            for column in order_by
        ]
        sql += f'\nORDER BY {", ".join(cleaned_order)}'

    if limit is not None:
        sql += f"\nLIMIT {int(limit)}"

    dataframe = read_sql_dataframe(
        engine,
        sql,
        params=params or {},
    ).copy()

    if "timestamp" in dataframe.columns:
        dataframe["timestamp"] = pd.to_datetime(dataframe["timestamp"], errors="coerce")

    if "machine_status" in dataframe.columns:
        dataframe["machine_status"] = (
            dataframe["machine_status"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    return dataframe

In [35]:
def build_kaggle_style_comparison_frame(
    dataframe: pd.DataFrame,
    *,
    timestamp_column: str = "timestamp",
    status_column: str = "machine_status",
    sensor_prefix: str = "sensor_",
    keep_identity_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    """
    Reduce a dataframe to a comparison-ready shape:
    identity cols + timestamp + sensors + machine_status.
    """
    keep_identity_columns = list(keep_identity_columns or [])

    sensor_columns = sorted(
        [column for column in dataframe.columns if str(column).startswith(sensor_prefix)]
    )

    columns: list[str] = []
    for column in keep_identity_columns:
        if column in dataframe.columns:
            columns.append(column)

    if timestamp_column in dataframe.columns:
        columns.append(timestamp_column)

    columns.extend(sensor_columns)

    if status_column in dataframe.columns:
        columns.append(status_column)

    out = dataframe.loc[:, columns].copy()

    if timestamp_column in out.columns:
        out[timestamp_column] = pd.to_datetime(out[timestamp_column], errors="coerce")

    if status_column in out.columns:
        out[status_column] = out[status_column].astype("string").str.strip().str.upper()

    return out

In [36]:
def _get_sensor_columns(
    dataframe: pd.DataFrame,
    *,
    sensor_prefix: str = "sensor_",
) -> list[str]:
    return sorted([column for column in dataframe.columns if str(column).startswith(sensor_prefix)])


def _safe_corr(
    dataframe: pd.DataFrame,
    sensor_a: str,
    sensor_b: str,
) -> float:
    if sensor_a not in dataframe.columns or sensor_b not in dataframe.columns:
        return np.nan

    subset = dataframe[[sensor_a, sensor_b]].dropna()
    if len(subset) < 3:
        return np.nan

    return float(subset[sensor_a].corr(subset[sensor_b]))


def _cluster_average_correlation(
    dataframe: pd.DataFrame,
    cluster: Sequence[str],
) -> float:
    sensors = [sensor for sensor in cluster if sensor in dataframe.columns]
    if len(sensors) < 2:
        return np.nan

    values = []
    for sensor_a, sensor_b in combinations(sensors, 2):
        corr_value = _safe_corr(dataframe, sensor_a, sensor_b)
        if np.isfinite(corr_value):
            values.append(corr_value)

    if not values:
        return np.nan

    return float(np.mean(values))


def _global_correlation_error(
    source_df: pd.DataFrame,
    synthetic_df: pd.DataFrame,
    *,
    sensor_columns: Sequence[str],
) -> dict[str, float]:
    common_sensors = [sensor for sensor in sensor_columns if sensor in source_df.columns and sensor in synthetic_df.columns]
    if len(common_sensors) < 2:
        return {
            "pair_count": 0,
            "mean_abs_diff": np.nan,
            "median_abs_diff": np.nan,
            "p90_abs_diff": np.nan,
        }

    diffs = []
    for sensor_a, sensor_b in combinations(common_sensors, 2):
        src_corr = _safe_corr(source_df, sensor_a, sensor_b)
        syn_corr = _safe_corr(synthetic_df, sensor_a, sensor_b)
        if np.isfinite(src_corr) and np.isfinite(syn_corr):
            diffs.append(abs(src_corr - syn_corr))

    if not diffs:
        return {
            "pair_count": 0,
            "mean_abs_diff": np.nan,
            "median_abs_diff": np.nan,
            "p90_abs_diff": np.nan,
        }

    diffs_array = np.asarray(diffs, dtype=float)

    return {
        "pair_count": int(diffs_array.size),
        "mean_abs_diff": float(np.mean(diffs_array)),
        "median_abs_diff": float(np.median(diffs_array)),
        "p90_abs_diff": float(np.percentile(diffs_array, 90)),
    }

In [37]:
def build_synthetic_vs_source_scorecard(
    *,
    source_df: pd.DataFrame,
    synthetic_df: pd.DataFrame,
    previous_synthetic_df: Optional[pd.DataFrame] = None,
    source_status_column: str = "machine_status",
    synthetic_status_column: str = "machine_status",
    previous_synthetic_status_column: Optional[str] = None,
    sensor_prefix: str = "sensor_",
    focus_sensors: Optional[Sequence[str]] = None,
    focus_pairs: Optional[Sequence[tuple[str, str]]] = None,
    focus_clusters: Optional[dict[str, Sequence[str]]] = None,
) -> dict[str, pd.DataFrame]:
    """
    Compare source vs synthetic and return a scorecard dict of DataFrames.

    Supports different status-column names across source/synthetic/previous synthetic.
    """
    if focus_sensors is None:
        focus_sensors = ["sensor_15", "sensor_50", "sensor_51", "sensor_06", "sensor_07", "sensor_08", "sensor_09"]

    if focus_pairs is None:
        focus_pairs = [
            ("sensor_08", "sensor_09"),
            ("sensor_14", "sensor_16"),
            ("sensor_17", "sensor_18"),
            ("sensor_19", "sensor_20"),
            ("sensor_20", "sensor_21"),
            ("sensor_22", "sensor_23"),
            ("sensor_25", "sensor_26"),
            ("sensor_31", "sensor_32"),
            ("sensor_32", "sensor_33"),
            ("sensor_34", "sensor_35"),
            ("sensor_35", "sensor_36"),
        ]

    if focus_clusters is None:
        focus_clusters = {
            "cluster_19_25": ["sensor_19", "sensor_20", "sensor_21", "sensor_22", "sensor_23", "sensor_24", "sensor_25"],
            "cluster_31_33": ["sensor_31", "sensor_32", "sensor_33"],
            "cluster_34_36": ["sensor_34", "sensor_35", "sensor_36"],
        }

    if previous_synthetic_status_column is None:
        previous_synthetic_status_column = synthetic_status_column

    sensor_columns = sorted(
        set(_get_sensor_columns(source_df, sensor_prefix=sensor_prefix))
        & set(_get_sensor_columns(synthetic_df, sensor_prefix=sensor_prefix))
    )

    # -------------------------
    # state mix
    # -------------------------
    source_states = set(
        source_df[source_status_column].dropna().astype(str).str.upper().unique()
    )
    synthetic_states = set(
        synthetic_df[synthetic_status_column].dropna().astype(str).str.upper().unique()
    )

    state_values = sorted(source_states.union(synthetic_states))

    state_rows = []
    source_n = max(len(source_df), 1)
    synthetic_n = max(len(synthetic_df), 1)

    for state_value in state_values:
        src_count = int((source_df[source_status_column].astype(str).str.upper() == state_value).sum())
        syn_count = int((synthetic_df[synthetic_status_column].astype(str).str.upper() == state_value).sum())

        row = {
            "state": state_value,
            "source_count": src_count,
            "synthetic_count": syn_count,
            "source_pct": src_count / source_n,
            "synthetic_pct": syn_count / synthetic_n,
            "pct_diff": (syn_count / synthetic_n) - (src_count / source_n),
        }

        if previous_synthetic_df is not None:
            prev_n = max(len(previous_synthetic_df), 1)
            prev_count = int(
                (previous_synthetic_df[previous_synthetic_status_column].astype(str).str.upper() == state_value).sum()
            )
            row["previous_synthetic_count"] = prev_count
            row["previous_synthetic_pct"] = prev_count / prev_n
            row["synthetic_pct_change_vs_previous"] = (syn_count / synthetic_n) - (prev_count / prev_n)

        state_rows.append(row)

    state_mix_df = pd.DataFrame(state_rows)

    # -------------------------
    # missingness
    # -------------------------
    missingness_rows = []
    for sensor in focus_sensors:
        if sensor not in source_df.columns and sensor not in synthetic_df.columns:
            continue

        src_missing = float(source_df[sensor].isna().mean() * 100) if sensor in source_df.columns else np.nan
        syn_missing = float(synthetic_df[sensor].isna().mean() * 100) if sensor in synthetic_df.columns else np.nan

        row = {
            "sensor": sensor,
            "source_missing_pct": src_missing,
            "synthetic_missing_pct": syn_missing,
            "missing_pct_diff": syn_missing - src_missing,
        }

        if previous_synthetic_df is not None and sensor in previous_synthetic_df.columns:
            prev_missing = float(previous_synthetic_df[sensor].isna().mean() * 100)
            row["previous_synthetic_missing_pct"] = prev_missing
            row["synthetic_missing_pct_change_vs_previous"] = syn_missing - prev_missing

        missingness_rows.append(row)

    missingness_df = pd.DataFrame(missingness_rows)

    # -------------------------
    # focus pairs
    # -------------------------
    pair_rows = []
    for sensor_a, sensor_b in focus_pairs:
        src_corr = _safe_corr(source_df, sensor_a, sensor_b)
        syn_corr = _safe_corr(synthetic_df, sensor_a, sensor_b)

        row = {
            "sensor_a": sensor_a,
            "sensor_b": sensor_b,
            "source_corr": src_corr,
            "synthetic_corr": syn_corr,
            "abs_diff_vs_source": abs(syn_corr - src_corr) if np.isfinite(src_corr) and np.isfinite(syn_corr) else np.nan,
        }

        if previous_synthetic_df is not None:
            prev_corr = _safe_corr(previous_synthetic_df, sensor_a, sensor_b)
            row["previous_synthetic_corr"] = prev_corr
            row["synthetic_corr_change_vs_previous"] = syn_corr - prev_corr if np.isfinite(prev_corr) and np.isfinite(syn_corr) else np.nan

        pair_rows.append(row)

    focus_pairs_df = pd.DataFrame(pair_rows)

    # -------------------------
    # focus clusters
    # -------------------------
    cluster_rows = []
    for cluster_name, cluster_sensors in focus_clusters.items():
        src_avg = _cluster_average_correlation(source_df, cluster_sensors)
        syn_avg = _cluster_average_correlation(synthetic_df, cluster_sensors)

        row = {
            "cluster_name": cluster_name,
            "cluster_sensors": list(cluster_sensors),
            "source_avg_corr": src_avg,
            "synthetic_avg_corr": syn_avg,
            "abs_diff_vs_source": abs(syn_avg - src_avg) if np.isfinite(src_avg) and np.isfinite(syn_avg) else np.nan,
        }

        if previous_synthetic_df is not None:
            prev_avg = _cluster_average_correlation(previous_synthetic_df, cluster_sensors)
            row["previous_synthetic_avg_corr"] = prev_avg
            row["synthetic_avg_corr_change_vs_previous"] = syn_avg - prev_avg if np.isfinite(prev_avg) and np.isfinite(syn_avg) else np.nan

        cluster_rows.append(row)

    focus_clusters_df = pd.DataFrame(cluster_rows)

    # -------------------------
    # global correlation error
    # -------------------------
    global_error = _global_correlation_error(
        source_df,
        synthetic_df,
        sensor_columns=sensor_columns,
    )
    global_error_row = dict(global_error)

    if previous_synthetic_df is not None:
        prev_global_error = _global_correlation_error(
            source_df,
            previous_synthetic_df,
            sensor_columns=sensor_columns,
        )
        global_error_row["previous_mean_abs_diff"] = prev_global_error["mean_abs_diff"]
        global_error_row["previous_median_abs_diff"] = prev_global_error["median_abs_diff"]
        global_error_row["previous_p90_abs_diff"] = prev_global_error["p90_abs_diff"]
        global_error_row["mean_abs_diff_change_vs_previous"] = global_error["mean_abs_diff"] - prev_global_error["mean_abs_diff"]

    global_correlation_error_df = pd.DataFrame([global_error_row])

    return {
        "state_mix": state_mix_df,
        "missingness": missingness_df,
        "focus_pairs": focus_pairs_df,
        "focus_clusters": focus_clusters_df,
        "global_correlation_error": global_correlation_error_df,
    }

In [38]:
source_df = load_source_dataframe_for_comparison(
    file_name="pump_sensor_data/sensor.csv",  
    use_data_raw_dir=True,
)

In [54]:
synthetic_df = read_synthetic_comparison_projection_dataframe(
    schema="capstone",
    table_name="synthetic_observations_timestamped_stage",
)

In [55]:
source_compare_df = build_kaggle_style_comparison_frame(
    source_df,
    timestamp_column="timestamp",
    status_column="machine_status",
)

synthetic_compare_df = synthetic_df.copy()


In [ ]:
# =========================================================
# Build validation target subsets
# =========================================================
# After clean-normal profiling, synthetic NORMAL should be judged primarily
# against Silver normal_clean, not the full Kaggle NORMAL population.
# Full Kaggle NORMAL remains a diagnostic comparison only.
#
# This cell resolves the Silver profiled dataframe from:
#   1. latest silver_subsets truth record
#   2. common direct artifact paths
#   3. recursive artifact search
# =========================================================

DATASET_NAME = "pump"
SILVER_SUBSETS_LAYER_NAME = "silver_subsets"

silver_subsets_artifact_dir = (
    paths.artifacts
    / SILVER_SUBSETS_LAYER_NAME
    / DATASET_NAME
)

truth_index_path = paths.artifacts / "truths" / "truth_index.jsonl"
truth_dir = paths.artifacts / "truths"


def resolve_latest_truth_record(
    *,
    truth_index_path: Path,
    truth_dir: Path,
    layer_name: str,
    dataset_name: str,
) -> dict | None:
    """
    Resolve the latest truth record for a layer/dataset from truth_index.jsonl.
    Returns None if unavailable.
    """
    if not truth_index_path.exists():
        print(f"WARNING: truth_index does not exist: {truth_index_path}")
        return None

    truth_index_df = pd.read_json(truth_index_path, lines=True)

    if truth_index_df.empty:
        print(f"WARNING: truth_index is empty: {truth_index_path}")
        return None

    matches = truth_index_df.loc[
        truth_index_df["layer_name"].astype(str).eq(layer_name)
        & truth_index_df["dataset_name"].astype(str).eq(dataset_name)
    ].copy()

    if matches.empty:
        print(
            f"WARNING: No truth_index rows found for "
            f"layer={layer_name!r}, dataset={dataset_name!r}"
        )
        return None

    latest_row = matches.tail(1).iloc[0]
    truth_hash = str(latest_row["truth_hash"])

    candidate_paths = list(
        (truth_dir / layer_name).rglob(f"*{truth_hash}*.json")
    )

    if not candidate_paths:
        print(f"WARNING: Could not find truth JSON for hash={truth_hash}")
        return None

    truth_path = sorted(candidate_paths)[-1]

    with open(truth_path, "r", encoding="utf-8") as f:
        truth_record = json.load(f)

    print("Resolved latest silver_subsets truth:", truth_path)
    return truth_record


def resolve_silver_profiled_dataframe_path() -> Path:
    """
    Resolve profiled dataframe path for validation.
    """

    # -----------------------------------------------------
    # 1. Truth-first resolution
    # -----------------------------------------------------
    truth_record = resolve_latest_truth_record(
        truth_index_path=truth_index_path,
        truth_dir=truth_dir,
        layer_name=SILVER_SUBSETS_LAYER_NAME,
        dataset_name=DATASET_NAME,
    )

    if truth_record is not None:
        artifact_paths = truth_record.get("artifact_paths", {}) or {}

        for key in [
            "profiled_dataframe_path",
            "silver_subsets_profiled_dataframe_path",
            "source_profiled_dataframe_path",
        ]:
            raw_path = artifact_paths.get(key)

            if raw_path:
                candidate = Path(raw_path)

                if candidate.exists():
                    print(f"Using profiled dataframe from truth artifact key: {key}")
                    return candidate

                print(f"WARNING: Truth path for {key} does not exist: {candidate}")

    # -----------------------------------------------------
    # 2. Common direct candidates
    # -----------------------------------------------------
    direct_candidates = [
        silver_subsets_artifact_dir / f"{DATASET_NAME}__silver_subsets__profiled_dataframe.parquet",
        silver_subsets_artifact_dir / "profiled_dataframe.parquet",
        silver_subsets_artifact_dir / "subsets" / f"{DATASET_NAME}__silver_subsets__profiled_dataframe.parquet",
        silver_subsets_artifact_dir / "data" / f"{DATASET_NAME}__silver_subsets__profiled_dataframe.parquet",
    ]

    for candidate in direct_candidates:
        if candidate.exists():
            print("Using profiled dataframe direct candidate:", candidate)
            return candidate

    # -----------------------------------------------------
    # 3. Recursive fallback search
    # -----------------------------------------------------
    search_matches = sorted(
        list(silver_subsets_artifact_dir.rglob("*profiled*dataframe*.parquet"))
        + list(silver_subsets_artifact_dir.rglob("*profiled*.parquet"))
    )

    if search_matches:
        print("Using profiled dataframe recursive match:", search_matches[0])
        print("All profiled dataframe matches:")
        for path in search_matches:
            print("  -", path)
        return search_matches[0]

    raise FileNotFoundError(
        "Could not resolve Silver profiled dataframe.\n"
        f"Checked direct candidates:\n"
        + "\n".join(str(path) for path in direct_candidates)
        + f"\n\nAlso searched recursively under:\n{silver_subsets_artifact_dir}\n\n"
        "Fix: rerun Silver 02a to regenerate the profiled dataframe, then rerun "
        "Silver 02b final truth/export cells."
    )


SILVER_SUBSETS_PROFILED_PATH = resolve_silver_profiled_dataframe_path()

silver_profiled_df = pd.read_parquet(SILVER_SUBSETS_PROFILED_PATH)

STATE_COL_PROFILED = "machine_status__profiled"
STATE_COL_SYNTHETIC = "machine_status"

print("SILVER_SUBSETS_PROFILED_PATH:", SILVER_SUBSETS_PROFILED_PATH)
print("silver_profiled_df shape:", silver_profiled_df.shape)


# -----------------------------
# Source target: clean normal
# -----------------------------
normal_clean_source_df = silver_profiled_df.loc[
    silver_profiled_df[STATE_COL_PROFILED]
    .astype(str)
    .str.lower()
    .eq("normal_clean")
].copy()

normal_clean_source_compare_df = build_kaggle_style_comparison_frame(
    normal_clean_source_df,
    timestamp_column="timestamp",
    status_column=STATE_COL_PROFILED,
)

# Give the scorecard a comparable state label.
normal_clean_source_compare_df["machine_status"] = "NORMAL"


# -----------------------------
# Source target: suspect / contaminated normal
# -----------------------------
suspect_normal_source_df = silver_profiled_df.loc[
    silver_profiled_df[STATE_COL_PROFILED]
    .astype(str)
    .str.lower()
    .isin(["normal_suspect", "normal_contaminated"])
].copy()

suspect_normal_source_compare_df = build_kaggle_style_comparison_frame(
    suspect_normal_source_df,
    timestamp_column="timestamp",
    status_column=STATE_COL_PROFILED,
)

suspect_normal_source_compare_df["machine_status"] = "NORMAL"


# -----------------------------
# Diagnostic target: full Kaggle NORMAL
# -----------------------------
full_kaggle_normal_compare_df = source_compare_df.loc[
    source_compare_df["machine_status"]
    .astype(str)
    .str.upper()
    .eq("NORMAL")
].copy()


# -----------------------------
# Synthetic NORMAL
# -----------------------------
synthetic_normal_compare_df = synthetic_compare_df.loc[
    synthetic_compare_df[STATE_COL_SYNTHETIC]
    .astype(str)
    .str.upper()
    .eq("NORMAL")
].copy()


# -----------------------------
# Synthetic buildup phase
# -----------------------------
if "phase" in synthetic_compare_df.columns:
    synthetic_buildup_compare_df = synthetic_compare_df.loc[
        synthetic_compare_df["phase"]
        .astype(str)
        .str.lower()
        .eq("buildup")
    ].copy()

    synthetic_buildup_compare_df["machine_status"] = "NORMAL"
else:
    synthetic_buildup_compare_df = pd.DataFrame()


print("normal_clean_source_compare_df:", normal_clean_source_compare_df.shape)
print("suspect_normal_source_compare_df:", suspect_normal_source_compare_df.shape)
print("full_kaggle_normal_compare_df:", full_kaggle_normal_compare_df.shape)
print("synthetic_normal_compare_df:", synthetic_normal_compare_df.shape)
print("synthetic_buildup_compare_df:", synthetic_buildup_compare_df.shape)

In [48]:
previous_synthetic_compare_df = None
# or load another synthetic table/export if you want run-vs-run comparison

In [56]:
print("source_compare_df columns:")
print(source_compare_df.columns.tolist())

print("\nsynthetic_compare_df columns:")
print(synthetic_compare_df.columns.tolist())

source_compare_df columns:
['timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51', 'machine_status']

synthetic_compare_df columns:
['timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16',

In [57]:
print("source status columns:", [c for c in source_compare_df.columns if "status" in c.lower() or "state" in c.lower()])
print("synthetic status columns:", [c for c in synthetic_compare_df.columns if "status" in c.lower() or "state" in c.lower()])

source status columns: ['machine_status']
synthetic status columns: ['machine_status']


In [ ]:
# =========================================================
# Build separated scorecards
# =========================================================

scorecard_targets = [
    {
        "label": "synthetic_vs_normal_clean",
        "source_df": normal_clean_source_compare_df,
        "synthetic_df": synthetic_normal_compare_df,
        "source_status_column": "machine_status",
        "synthetic_status_column": "machine_status",
    },
    {
        "label": "synthetic_vs_full_normal",
        "source_df": full_kaggle_normal_compare_df,
        "synthetic_df": synthetic_normal_compare_df,
        "source_status_column": "machine_status",
        "synthetic_status_column": "machine_status",
    },
    {
        "label": "synthetic_vs_full_source",
        "source_df": source_compare_df,
        "synthetic_df": synthetic_compare_df,
        "source_status_column": "machine_status",
        "synthetic_status_column": "machine_status",
    },
]

if not synthetic_buildup_compare_df.empty and not suspect_normal_source_compare_df.empty:
    scorecard_targets.append(
        {
            "label": "synthetic_buildup_vs_suspect_normal",
            "source_df": suspect_normal_source_compare_df,
            "synthetic_df": synthetic_buildup_compare_df,
            "source_status_column": "machine_status",
            "synthetic_status_column": "machine_status",
        }
    )

scorecards = {}

for target in scorecard_targets:
    label = target["label"]

    print(f"Building scorecard: {label}")
    print("  source:", target["source_df"].shape)
    print("  synthetic:", target["synthetic_df"].shape)

    scorecards[label] = build_synthetic_vs_source_scorecard(
        source_df=target["source_df"],
        synthetic_df=target["synthetic_df"],
        previous_synthetic_df=previous_synthetic_compare_df,
        source_status_column=target["source_status_column"],
        synthetic_status_column=target["synthetic_status_column"],
        previous_synthetic_status_column="machine_status",
    )

# Backward-compatible alias so older display/export code does not immediately break.
scorecard = scorecards["synthetic_vs_full_source"]

print("Built scorecards:", list(scorecards.keys()))

In [ ]:
# =========================================================
# Display separated scorecard summaries
# =========================================================

for label, scorecard_dict in scorecards.items():
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)

    print("\nState mix:")
    display(scorecard_dict["state_mix"])

    print("\nMissingness:")
    display(scorecard_dict["missingness"])

    print("\nFocus pairs:")
    display(scorecard_dict["focus_pairs"])

    print("\nFocus clusters:")
    display(scorecard_dict["focus_clusters"])

    print("\nGlobal correlation error:")
    display(scorecard_dict["global_correlation_error"])

,state,source_count,synthetic_count,source_pct,synthetic_pct,pct_diff
0,BROKEN,7,11,0.000032,0.000049,0.000017
1,NORMAL,205836,211465,0.934259,0.939844,0.005585
2,RECOVERING,14477,13524,0.065709,0.060107,-0.005602


,sensor,source_missing_pct,synthetic_missing_pct,missing_pct_diff
0,sensor_15,100.000000,100.000000,0.000000
1,sensor_50,34.956881,34.960000,0.003119
2,sensor_51,6.982117,6.985778,0.003661
3,sensor_06,2.177741,2.181333,0.003592
4,sensor_07,2.474129,2.475556,0.001427
5,sensor_08,2.317992,2.317778,-0.000214
6,sensor_09,2.085603,2.085778,0.000175


,sensor_a,sensor_b,source_corr,synthetic_corr,abs_diff_vs_source
0,sensor_08,sensor_09,0.844928,0.220807,0.624121
1,sensor_14,sensor_16,0.990359,0.563216,0.427143
2,sensor_17,sensor_18,0.983326,0.205718,0.777608
3,sensor_19,sensor_20,0.998227,0.332573,0.665654
4,sensor_20,sensor_21,0.997075,0.795261,0.201815
5,sensor_22,sensor_23,0.986869,0.707011,0.279857
6,sensor_25,sensor_26,0.970675,0.442162,0.528512
7,sensor_31,sensor_32,0.865486,0.824190,0.041296
8,sensor_32,sensor_33,0.854665,0.800040,0.054625
9,sensor_34,sensor_35,0.810734,0.133583,0.677151


,cluster_name,cluster_sensors,source_avg_corr,synthetic_avg_corr,abs_diff_vs_source
0,cluster_19_25,"[sensor_19, sensor_20, sensor_21, sensor_22, s...",0.983080,0.642225,0.340855
1,cluster_31_33,"[sensor_31, sensor_32, sensor_33]",0.855524,0.756935,0.098588
2,cluster_34_36,"[sensor_34, sensor_35, sensor_36]",0.774346,0.161076,0.613269


,pair_count,mean_abs_diff,median_abs_diff,p90_abs_diff
0,1275,0.22302,0.14735,0.643898


In [60]:
def export_scorecard_tables(
    *,
    scorecard: dict[str, pd.DataFrame],
    output_dir: str | Path,
    filename_prefix: str = "synthetic_scorecard",
) -> dict[str, str]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    out_paths: dict[str, str] = {}

    for table_name, dataframe in scorecard.items():
        path = output_dir / f"{filename_prefix}__{table_name}.csv"
        dataframe.to_csv(path, index=False)
        out_paths[table_name] = str(path)

    return out_paths

In [ ]:
# =========================================================
# Export separated scorecards
# =========================================================

paths = get_paths()

all_scorecard_export_paths = {}

for label, scorecard_dict in scorecards.items():
    export_paths = export_scorecard_tables(
        scorecard=scorecard_dict,
        output_dir=paths.data_synthetic,
        filename_prefix=label,
    )

    all_scorecard_export_paths[label] = export_paths

all_scorecard_export_paths

{'state_mix': '/workspace/data/synthetic/synthetic_vs_source_scorecard__state_mix.csv',
 'missingness': '/workspace/data/synthetic/synthetic_vs_source_scorecard__missingness.csv',
 'focus_pairs': '/workspace/data/synthetic/synthetic_vs_source_scorecard__focus_pairs.csv',
 'focus_clusters': '/workspace/data/synthetic/synthetic_vs_source_scorecard__focus_clusters.csv',
 'global_correlation_error': '/workspace/data/synthetic/synthetic_vs_source_scorecard__global_correlation_error.csv'}

In [ ]:
engine.dispose()

In [ ]:
current_datetime = datetime.now()
adjusted_time = current_datetime - timedelta(hours=4)

formatted_datetime = adjusted_time.strftime("%m%d%Y_%H%M")

In [ ]:
print(f"Score card complete at {formatted_datetime}")

In [ ]:
%store -r generation_started_current_datetime

generation_complete_current_datetime = datetime.now()
generation_completed_adjusted_time = generation_complete_current_datetime - timedelta(hours=4)

generation_completed_formatted_datetime = generation_completed_adjusted_time.strftime("%m%d%Y_%H%M")

total_runtime = generation_complete_current_datetime - generation_started_current_datetime


print(f"Synthetic Data Generation Completed at {generation_completed_formatted_datetime}")
print(f"Total runtime: {total_runtime}")